# DS15 · Feature selection and the full null pipeline

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PD04](../../curriculum/papers/design.md#pd04).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** When does selecting features become part of the model that must be evaluated independently?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Format:** 75–100 minutes of guided work, plus 30–60 minutes in the assigned existing course material. **Prerequisite:** the foundations notebooks; follow this strand in order. The core exercise is synthetic, offline, and independently runnable. It demonstrates mechanics, not a validated participant-data analysis.

## Existing course material

Read [Data 8: Training and testing; supplement sklearn leakage examples](https://github.com/data-8/textbook/blob/5235b7653f8dfaeb90e43419b9aa069322f2d60b/chapters/17/2/Training_and_Testing.ipynb). Use the indicated topic, then return here to apply it to a neuroimaging question. Berkeley material is linked in its original form, not adapted or redistributed; its CC BY-NC-ND terms remain upstream. Neuromatch material is CC BY 4.0 with separately licensed software; selected unmodified copies live in `third_party/data_science`. The explanation and dataset below are original.

## Understand the transformation

Selecting a feature because it correlates with the outcome is a learned operation. If all participants are used for this selection before train/test splitting, the test labels have already influenced the model. Subsequent cross-validation of only the final regressor does not undo that information leak. The whole selection procedure must be refitted inside each training fold.

The danger grows when many features compete. With thousands of random voxel-like variables, some will correlate strongly with an unrelated random outcome by chance. That best observed correlation is an optimistically selected statistic. The correct null experiment repeats selection, fitting, tuning, and evaluation under the null, rather than permuting labels after freezing the feature chosen using the original labels.

We compare a deliberately circular in-sample selection with its behavior on independent data. The example is not a recommended performance estimator; it shows why feature-search breadth matters. A single test realization can fluctuate, so inspect repeated simulations or use an appropriate permutation distribution. For dependent participants or time series, arbitrary row permutation is invalid; preserve exchangeability blocks or use a justified alternative.

An AI should be asked to draw a boundary around every operation that learns from y, including screening thresholds, ROI selection, and tuning. Generalization is a property of the entire procedure, not just the classifier at its end. The modeling strand supplies grouped nested validation after this mechanical lesson.

## AI-guided prediction

First answer in your own words; then send this to Goose/Ollama or ChatGPT:

> Before any feature screening, ask which labels the selector is allowed to see. Select the strongest null feature in one sample, test that fixed feature in an independent sample, and explain why permuting only the final fit is insufficient for a full pipeline null.

Use the model as a tutor and snippet writer. Require it to name the axes, units, fitting population, expected output, and one failure check. A code cell that runs is not proof that it answers the scientific question. Keep raw data unchanged and save your actual settings.

## Experiment

Generate 2,000 random features and a random outcome. Select the highest absolute discovery correlation. Deliberate error: call that discovery correlation a held-out prediction score.

Run the following cells in order. Before each, predict what should remain unchanged and what should differ. The assertions test specific mathematical or bookkeeping properties, not clinical validity.

In [1]:
import numpy as np
rng=np.random.default_rng(315); n,p=60,2000
X=rng.normal(size=(n,p)); y=rng.normal(size=n)
Xc=X-X.mean(axis=0); yc=y-y.mean()
corr=(Xc.T@yc)/(np.linalg.norm(Xc,axis=0)*np.linalg.norm(yc))
j=np.argmax(abs(corr))
newX=rng.normal(size=(500,p)); newy=rng.normal(size=500)
rep=np.corrcoef(newX[:,j],newy)[0,1]
assert abs(corr[j])>.3
assert abs(rep)<.15
print('Chosen feature:',j,'selected discovery r:',corr[j],'independent r:',rep)
print('No features have a planted relationship in this simulation.')

Chosen feature: 1232 selected discovery r: -0.4130738282219884 independent r: -0.0067721396177212725
No features have a planted relationship in this simulation.


## Explain, break, transfer

1. Save an input → operation → output diagram and state what information was lost.
2. Make the specified wrong choice above. Compare its result with the reference checks; explain why the misleading result is possible.
3. Work through the assigned upstream chapter's example using its own environment or hosted reader. Record one difference between its data and a participant/voxel/time-series dataset.
4. Ask the AI for a short application to a real imaging table, but do not run it until participant identifiers, units, missingness, and any training/test boundary are explicit. Never infer those properties from the column names alone.

**Evidence to submit:** one labeled result, the changed parameter, a failure diagnosis, and a five-sentence interpretation that separates a computational check from the research claim. Explain the result without looking at the model's wording.

<details><summary>Instructor check / answer guide</summary>

The selected discovery correlation is inflated by the search. The same feature has no planted relationship in replication. The exact replication correlation is random, not guaranteed to be exactly zero.

</details>

**Scope:** This local notebook and its numerical checks are part of the executable core. Completion of the external chapter is a learner assignment; its execution is not implied by the local result. No endorsement by the source authors or USC is implied.

### Return to the research question

Reopen [PD04](../../curriculum/papers/design.md#pd04) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
